# Notebook 6 — Self-Correcting Code Execution Agent

This notebook builds an agent that does not just generate answers — it takes **actions**, observes what happens, and **fixes its own mistakes**.

Every previous notebook retrieved information and asked an LLM to answer from it. This notebook goes further. The agent writes Python code, actually runs it, looks at whether it worked or failed, and if it failed, it tries to fix the code and run it again.

This is the core loop of autonomous AI agents:

```
Generate → Execute → Observe → Correct → Retry → Verify
```

**How this differs from Notebook 5 (Agentic RAG):**

| Notebook 5 | Notebook 6 |
|-----------|-----------|
| Plans and retrieves information | Generates and executes code |
| Agent action = run a search query | Agent action = run Python code |
| Observation = retrieved text chunks | Observation = stdout or error message |
| No execution loop | Self-correction loop with retry |

**How to run:** run all cells from top to bottom. Use the Gradio interface to submit coding tasks.

**API key needed:** `GOOGLE_API_KEY` (Gemini) in Colab Secrets.

**Important note on safety:** this notebook runs generated code in a subprocess. It is not a fully isolated sandbox. For educational and experimental use in Colab, this is fine. For production systems that run untrusted code, use Docker or OS-level isolation.


In [ ]:
!pip install -q google-generativeai gradio

In [ ]:
import os
import re
import sys
import subprocess
import tempfile
import time
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

import google.generativeai as genai

print("All imports successful.")

All imports successful.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Execution Data Models

Every execution attempt is recorded as a structured object.

`ExecutionAttempt` stores:
- The code that was generated or corrected
- The output (stdout)
- The error message (stderr) if it failed
- The return code
- The attempt number
- Whether it succeeded

`ExecutionHistory` is a list of all attempts. This replaces a simple success/failure flag with a full log that you can inspect. You can see exactly what went wrong at each attempt and how the agent fixed it.

Using structured data models (Pydantic) here means the agent's working memory is always validated and well-typed.


In [ ]:
@dataclass
class ExecutionResult:
    """Raw result from the subprocess executor."""

    success: bool
    stdout: str
    stderr: str
    returncode: int
    elapsed_seconds: float


@dataclass
class ExecutionAttempt:
    """One complete attempt: code generated → executed → result recorded."""

    attempt_number: int
    code: str
    result: ExecutionResult
    correction_prompt: Optional[str] = None  # None on the first attempt


@dataclass
class ExecutionHistory:
    """Full record of all attempts for one task."""

    task: str
    attempts: List[ExecutionAttempt] = field(default_factory=list)
    final_output: Optional[str] = None
    succeeded: bool = False

    def add(self, attempt: ExecutionAttempt) -> None:
        self.attempts.append(attempt)

    def last_attempt(self) -> Optional[ExecutionAttempt]:
        return self.attempts[-1] if self.attempts else None

    def summary(self) -> str:
        """Human-readable summary of all attempts."""
        lines = [f"Task: {self.task}", f"Total attempts: {len(self.attempts)}", ""]
        for a in self.attempts:
            status = "✅ SUCCESS" if a.result.success else "❌ FAILED"
            lines.append(
                f"Attempt {a.attempt_number}  {status}  ({a.result.elapsed_seconds:.2f}s)"
            )
            lines.append(f"  Return code : {a.result.returncode}")
            if a.result.stdout.strip():
                lines.append(f"  stdout      : {a.result.stdout.strip()[:300]}")
            if a.result.stderr.strip():
                lines.append(f"  stderr      : {a.result.stderr.strip()[:300]}")
            lines.append("")
        if self.succeeded:
            lines.append(f"Final output:\n{self.final_output}")
        else:
            lines.append("All attempts failed.")
        return "\n".join(lines)


print("Data models defined.")

Data models defined.


## Gemini Setup

This cell connects to the Gemini API.

Gemini is used for three things in this notebook:
1. **Code generation** — converting a task description into Python code
2. **Self-correction** — fixing code that failed to run
3. **Result verification** — checking whether the output actually answers the task

Your API key is read from Colab Secrets (`GOOGLE_API_KEY`).


In [ ]:
def _load_api_key() -> str:
    """Load Gemini API key from Colab secrets, env var, or interactive prompt."""
    # 1. Colab secrets
    try:
        from google.colab import userdata

        key = userdata.get("GEMINI_API_KEY")
        if key and isinstance(key, str) and key.strip():
            return key.strip()
        # Handle dict responses from older Colab versions
        if isinstance(key, dict):
            for k in ("payload", "data"):
                if k in key:
                    v = key[k]
                    if isinstance(v, dict) and "payload" in v:
                        return str(v["payload"]).strip()
                    return str(v).strip()
    except Exception:
        pass

    # 2. Environment variable
    key = os.environ.get("GEMINI_API_KEY", "").strip()
    if key:
        return key

    # 3. Interactive fallback
    key = input("Enter your Gemini API key: ").strip()
    if not key:
        raise ValueError("No API key provided.")
    return key


API_KEY = _load_api_key()
genai.configure(api_key=API_KEY)
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini 2.5 Flash ready.")

## Code Extractor

Gemini sometimes wraps generated code in markdown code fences:

````
```python
def factorial(n):
    ...
```
````

This cell strips those fences and returns clean, runnable Python code.

It handles all common fence patterns so the executor always receives pure Python, regardless of how Gemini formats the response.


In [ ]:
def extract_python_code(text: str) -> str:
    """
    Extract clean Python code from a Gemini response.

    Handles:
    - ```python ... ```
    - ```  ... ```
    - Raw code with no fences
    """
    # Try fenced block first
    match = re.search(r"```(?:python)?\s*\n?(.*?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()

    # Fall back to stripping any remaining backtick lines
    lines = text.splitlines()
    clean = [l for l in lines if not l.strip().startswith("```")]
    return "\n".join(clean).strip()


# Quick test
raw = "```python\nprint('hello')\n```"
assert extract_python_code(raw) == "print('hello')"
print("Code extractor OK.")

## Subprocess Executor

This cell runs Python code in a child subprocess and captures the result.

**What it captures:**
- `stdout` — what the code printed
- `stderr` — any error messages or tracebacks
- `returncode` — 0 means success, anything else means failure
- `elapsed` — how long it took to run

**Design decisions:**
- Code is written to a temporary file so it never touches the working directory
- A timeout prevents the agent from getting stuck in an infinite loop
- The temporary file is always cleaned up, even if execution times out
- The function always returns a result — it never raises an exception

**Important:** this is subprocess execution, not OS-level sandboxing. Generated code can access the local filesystem and network. For Colab experiments this is acceptable. For production, wrap with Docker or gVisor.


In [ ]:
def run_subprocess(python_code: str, timeout: float = 10.0) -> ExecutionResult:
    """
    Write code to a temp file, run it in a subprocess, return a typed ExecutionResult.

    Args:
        python_code: valid Python source code as a string
        timeout:     wall-clock seconds before the process is killed

    Returns:
        ExecutionResult with success, stdout, stderr, returncode, elapsed_seconds
    """
    tmp_path: Optional[Path] = None
    t0 = time.monotonic()

    try:
        with tempfile.NamedTemporaryFile(
            suffix=".py", mode="w", encoding="utf-8", delete=False
        ) as f:
            f.write(python_code)
            tmp_path = Path(f.name)

        proc = subprocess.run(
            [sys.executable, str(tmp_path)],
            capture_output=True,
            text=True,
            timeout=timeout,
        )

        elapsed = time.monotonic() - t0
        return ExecutionResult(
            success=proc.returncode == 0,
            stdout=proc.stdout,
            stderr=proc.stderr,
            returncode=proc.returncode,
            elapsed_seconds=round(elapsed, 3),
        )

    except subprocess.TimeoutExpired:
        elapsed = time.monotonic() - t0
        return ExecutionResult(
            success=False,
            stdout="",
            stderr=f"TimeoutError: execution exceeded {timeout}s limit.",
            returncode=-1,
            elapsed_seconds=round(elapsed, 3),
        )

    except Exception as e:
        elapsed = time.monotonic() - t0
        return ExecutionResult(
            success=False,
            stdout="",
            stderr=f"ExecutorError: {e}",
            returncode=-2,
            elapsed_seconds=round(elapsed, 3),
        )

    finally:
        if tmp_path and tmp_path.exists():
            tmp_path.unlink(missing_ok=True)


# Smoke tests
r1 = run_subprocess("print('hello from executor')")
assert r1.success and "hello" in r1.stdout
print("✅ Success case OK:", r1.stdout.strip())

r2 = run_subprocess("x = [1, 2]\nprint(x[10])")
assert not r2.success and "IndexError" in r2.stderr
print("✅ Error case OK:", r2.stderr.strip().splitlines()[-1])

r3 = run_subprocess("import time; time.sleep(99)", timeout=1.0)
assert not r3.success and "Timeout" in r3.stderr
print("✅ Timeout case OK:", r3.stderr.strip())

## Code Generation Agent

This is the first stage of the pipeline.

The agent takes a plain English task and asks Gemini to write a self-contained Python script.

The prompt tells Gemini to:
- Write code that can run on its own (no missing imports)
- Use only the standard library unless the task specifically asks for something else
- Always include a `print()` call so there is always output to inspect
- Wrap logic in a `main()` function with a `if __name__ == "__main__"` guard

Example:

**Task:** "Calculate the factorial of 5"

**Generated code:**
```python
def factorial(n):
    if n == 0:
        return 1
    return n * factorial(n - 1)

if __name__ == "__main__":
    print(factorial(5))
```

The generated code is then executed — not returned to the user yet.


In [ ]:
def generate_code(task: str) -> str:
    """
    Ask Gemini to write a self-contained Python script for the given task.

    Returns:
        Clean Python code as a string (no markdown fences).
    """
    prompt = f"""Write a self-contained Python script to complete the following task.

TASK:
{task}

RULES:
1. Use only Python standard library unless the task explicitly requires a third-party package.
2. Wrap logic in a main() function.
3. End with:  if __name__ == "__main__": main()
4. Always produce at least one print() call so the output is observable.
5. Return ONLY the Python code. No explanations, no markdown fences, no comments outside the code.
"""

    response = gemini_model.generate_content(prompt)
    return extract_python_code(response.text)


# Quick test (we'll run it properly in the full loop)
test_code = generate_code("Print the first 5 Fibonacci numbers.")
print("Generated code preview:")
print(test_code[:300])

## Self-Correction Agent

When execution fails, this agent tries to fix the code.

It receives three things:
1. The original task (so it never loses the goal)
2. The code that failed
3. The full error traceback

It returns a corrected version of the script.

**Example:**

Failed code:
```python
print(factorial(5))  # factorial not defined
```

Error: `NameError: name 'factorial' is not defined`

Corrected code:
```python
def factorial(n):
    if n == 0:
        return 1
    return n * factorial(n - 1)

print(factorial(5))
```

Separating code generation and self-correction into two distinct functions makes the pipeline easier to test and extend independently.


In [ ]:
def generate_correction(
    task: str, failed_code: str, error_message: str, attempt_number: int
) -> str:
    """
    Ask Gemini to fix a failed script given the traceback.

    Args:
        task:           original natural language task
        failed_code:    the script that failed
        error_message:  stderr / traceback from the subprocess
        attempt_number: which correction attempt this is (for context)

    Returns:
        Clean corrected Python code as a string.
    """
    prompt = f"""A Python script failed during execution. Fix it.

ORIGINAL TASK:
{task}

FAILED SCRIPT (Attempt {attempt_number}):
{failed_code}

EXECUTION ERROR:
{error_message}

INSTRUCTIONS:
1. Carefully read the error traceback above.
2. Identify the root cause.
3. Rewrite the COMPLETE corrected script — do not omit any logic.
4. Keep all the same requirements from the original task.
5. Return ONLY the corrected Python code. No explanations, no markdown fences.
"""

    response = gemini_model.generate_content(prompt)
    return extract_python_code(response.text)


print("Self-correction agent defined.")

## Full Self-Correcting Agent Loop

`run_self_correcting_agent()` ties everything together.

```
Task
  ↓
generate_code()          → attempt 1
  ↓
run_subprocess()
  ↓ success? → done
  ↓ fail?
generate_correction()    → attempt 2
  ↓
run_subprocess()
  ↓ success? → done
  ↓ fail?
... (up to max_attempts)
  ↓
ExecutionHistory (all attempts recorded)
```

Every attempt — the code, the output, the error — is stored in the execution history. No state is thrown away. The full correction trace is available for inspection.

The maximum number of attempts is configurable. By default it tries up to 3 times before stopping.


In [ ]:
def run_self_correcting_agent(
    task: str, max_attempts: int = 3, timeout: float = 10.0
) -> ExecutionHistory:
    """
    Run the self-correcting code execution agent.

    Args:
        task:         natural language description of what the code should do
        max_attempts: maximum number of generate-execute-correct cycles
        timeout:      per-execution subprocess timeout in seconds

    Returns:
        ExecutionHistory with all attempts recorded and final_output set on success.
    """
    history = ExecutionHistory(task=task)

    current_code: Optional[str] = None
    last_error: Optional[str] = None

    for attempt_num in range(1, max_attempts + 1):

        # Generate or correct
        if attempt_num == 1:
            print(f"[Attempt {attempt_num}] Generating initial code...")
            current_code = generate_code(task)
        else:
            print(
                f"[Attempt {attempt_num}] Generating correction (error: {last_error.strip().splitlines()[-1] if last_error else 'unknown'})..."
            )
            current_code = generate_correction(
                task=task,
                failed_code=current_code,
                error_message=last_error,
                attempt_number=attempt_num - 1,
            )

        print(f"  Code ({len(current_code)} chars):")
        for line in current_code.splitlines()[:6]:
            print(f"    {line}")
        if current_code.count("\n") >= 6:
            print("    ...")

        # Execute 
        result = run_subprocess(current_code, timeout=timeout)

        attempt = ExecutionAttempt(
            attempt_number=attempt_num,
            code=current_code,
            result=result,
            correction_prompt=None if attempt_num == 1 else last_error,
        )
        history.add(attempt)

        # Evaluate 
        if result.success:
            print(
                f"  ✅ Success on attempt {attempt_num}. Output: {result.stdout.strip()[:120]}"
            )
            history.succeeded = True
            history.final_output = result.stdout.strip()
            return history

        print(
            f"  ❌ Failed (returncode={result.returncode}): {result.stderr.strip().splitlines()[-1] if result.stderr.strip() else 'no stderr'}"
        )
        last_error = result.stderr

    # All attempts exhausted
    print(f"All {max_attempts} attempts failed.")
    history.final_output = (
        f"All {max_attempts} attempts failed. Last error:\n{last_error}"
    )
    return history


# Quick test
print("=" * 55)
print("TEST: First 5 Fibonacci numbers")
print("=" * 55)
test_history = run_self_correcting_agent(
    "Print the first 5 Fibonacci numbers, one per line.", max_attempts=3
)
print()
print(test_history.summary())

## Result Verification Agent

A subprocess that exits with return code 0 (success) does not always mean the answer is correct.

Example — this code runs successfully:
```python
print("wrong answer")
```

But it does not actually compute whatever the task asked for.

The verification agent checks whether the actual output matches the intent of the original task. It asks Gemini: "Given this task and this output, is the output correct?"

This adds a semantic check on top of execution success. It distinguishes between "ran without errors" and "produced the right answer."


In [ ]:
def verify_output(task: str, stdout: str) -> Tuple[bool, str]:
    """
    Ask Gemini whether the execution output correctly solves the original task.

    Args:
        task:   original natural language task
        stdout: captured stdout from the successful execution

    Returns:
        (verified: bool, explanation: str)
    """
    if not stdout.strip():
        return False, "Output is empty — cannot verify."

    prompt = f"""A Python script was executed to complete a task. Evaluate whether the output is correct.

TASK:
{task}

ACTUAL OUTPUT:
{stdout.strip()}

Answer with EXACTLY one of:
VERIFIED: <one-sentence explanation of why the output is correct>
INCORRECT: <one-sentence explanation of what is wrong>

No other text.
"""

    try:
        response = gemini_model.generate_content(prompt)
        text = response.text.strip()
        if text.upper().startswith("VERIFIED"):
            return True, text
        return False, text
    except Exception as e:
        return False, f"Verifier error: {e}"


# Test the verifier
print("Verifier test:")
ok, explanation = verify_output(
    "Print the first 5 Fibonacci numbers, one per line.",
    test_history.final_output or "",
)
print(f"  Verified : {ok}")
print(f"  Explanation: {explanation}")

## Complete Pipeline

`run_pipeline()` combines the two stages:

1. `run_self_correcting_agent()` — generate, execute, correct, retry
2. `verify_output()` — semantic check on the final output

It returns a tuple of `(history, verified, explanation)`:
- `history` — every execution attempt with full details
- `verified` — True if the output was semantically correct
- `explanation` — Gemini's reasoning about whether the output is correct

This gives you full transparency into what happened at every step.


In [ ]:
def run_pipeline(
    task: str, max_attempts: int = 3, timeout: float = 10.0
) -> Tuple[ExecutionHistory, bool, str]:
    """
    Full pipeline: self-correcting execution + result verification.

    Returns:
        history:     ExecutionHistory with all attempts
        verified:    True if output was semantically verified as correct
        explanation: verifier's explanation string
    """
    print("=" * 60)
    print(f"TASK: {task}")
    print("=" * 60)

    history = run_self_correcting_agent(
        task, max_attempts=max_attempts, timeout=timeout
    )

    verified = False
    explanation = "Execution failed — verification skipped."

    if history.succeeded and history.final_output:
        print("\nRunning result verifier...")
        verified, explanation = verify_output(task, history.final_output)
        print(f"Verification: {'✅ VERIFIED' if verified else '⚠️  INCORRECT'}")
        print(f"  {explanation}")

    return history, verified, explanation


# Full pipeline demo
history, verified, explanation = run_pipeline(
    "Generate a list of the first 10 prime numbers and print them as a comma-separated string."
)
print()
print("─" * 60)
print("EXECUTION HISTORY:")
print(history.summary())
print()
print(f"VERIFICATION: {'✅ VERIFIED' if verified else '⚠️  INCORRECT'}")
print(explanation)

## Gradio Interface

Run this cell to launch the interactive interface.

The interface has three panels:
- **Final Output** — what the code printed when it ran successfully
- **Verification** — whether the output actually answers the task (Gemini's judgment)
- **Execution History** — every attempt with code, errors, output, and timing

This gives full visibility into the agent's correction process. You can see exactly what failed, what was fixed, and how many attempts it took.

Example tasks to try:
- "Write a function to calculate factorial of 10"
- "Generate the first 20 Fibonacci numbers"
- "Find all prime numbers below 50"
- "Sort a list of strings alphabetically and print them"


In [ ]:
import gradio as gr


def run_agent_ui(task: str, max_attempts: int, timeout: float):
    """Gradio handler — runs the full pipeline and returns formatted strings."""
    if not task or not task.strip():
        return "Please enter a task.", "—", "—"

    try:
        history, verified, explanation = run_pipeline(
            task=task.strip(), max_attempts=int(max_attempts), timeout=float(timeout)
        )

        # Final output panel
        if history.succeeded:
            output_text = history.final_output or "(no stdout)"
        else:
            output_text = "All attempts failed. See Execution History for details."

        # Verification panel
        if history.succeeded:
            status = "✅ VERIFIED" if verified else "⚠️  INCORRECT"
            verification_text = f"{status}\n\n{explanation}"
        else:
            verification_text = "Verification skipped — execution did not succeed."

        # History panel
        history_text = history.summary()

        return output_text, verification_text, history_text

    except Exception as e:
        return f"Pipeline error: {e}", "—", "—"


with gr.Blocks(title="Notebook 5 — Self-Correcting Code Agent") as demo:

    gr.Markdown("""
# Self-Correcting Code Execution Agent
**Notebook 5** — Generate → Execute → Observe → Correct → Verify

Describe a Python task in plain English. The agent will write the code,
run it, fix any errors, and verify the output is semantically correct.
""")

    with gr.Row():
        with gr.Column(scale=3):
            task_box = gr.Textbox(
                label="Task Description",
                placeholder="e.g. Print the first 10 prime numbers as a comma-separated string",
                lines=3,
            )
        with gr.Column(scale=1):
            max_attempts_slider = gr.Slider(
                minimum=1, maximum=5, value=3, step=1, label="Max Correction Attempts"
            )
            timeout_slider = gr.Slider(
                minimum=2.0,
                maximum=30.0,
                value=10.0,
                step=1.0,
                label="Execution Timeout (seconds)",
            )

    run_btn = gr.Button("Run Agent", variant="primary")

    with gr.Row():
        output_box = gr.Textbox(
            label="Final Output (stdout)", lines=8, interactive=False, scale=2
        )
        verify_box = gr.Textbox(
            label="Result Verification", lines=8, interactive=False, scale=1
        )

    history_box = gr.Textbox(
        label="Execution History (all attempts)", lines=15, interactive=False
    )

    run_btn.click(
        fn=run_agent_ui,
        inputs=[task_box, max_attempts_slider, timeout_slider],
        outputs=[output_box, verify_box, history_box],
    )

    gr.Examples(
        examples=[
            ["Print the first 10 prime numbers as a comma-separated string.", 3, 10],
            ["Read the current UTC time and print it in ISO 8601 format.", 3, 10],
            ["Create a list of squares of numbers 1 to 15 and print their sum.", 3, 10],
            [
                "Sort the list ['banana', 'apple', 'cherry', 'date'] and print each item.",
                3,
                10,
            ],
            ["Print a 5x5 multiplication table, aligned in columns.", 3, 10],
            [
                "Count the vowels in the string 'self-correcting code agent' and print the result.",
                3,
                10,
            ],
        ],
        inputs=[task_box, max_attempts_slider, timeout_slider],
    )

demo.launch(share=True)

## Current Limitations and Future Improvements

**Current limitations:**

| Limitation | Detail |
|-----------|--------|
| Subprocess isolation only | Generated code can access filesystem and network. Use Docker for production. |
| Python only | The executor only runs Python. Other languages are not supported. |
| No output schema validation | Verifier uses Gemini judgment, not strict expected-output checks. |
| Stateless between sessions | Execution history is not saved when the runtime restarts. |
| Single agent | Generation, execution, correction, and verification all run sequentially. |

**Future improvements:**

| Enhancement | What it would add |
|------------|------------------|
| Docker sandbox | True OS-level isolation for generated code |
| Multi-language support | Run JavaScript, Bash, or SQL in separate executors |
| Structured output validation | Compare actual output against a typed expected schema |
| Persistent history | Save execution history to a file or database |
| RAG integration | Retrieve code examples from Notebook 5 before generating |
| Streaming output | Show subprocess output in real time in the Gradio interface |
| Adaptive timeout | Automatically increase timeout on timeout errors |
